<a href="https://colab.research.google.com/github/pris25123/Coordination-Collapse/blob/main/CoordinationCollapse_Full_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Coordination Tax: Coordination Collapse in Hierarchical Multi-Agent Code Generation

This notebook clones the Coordination-Collapse repository and runs the complete evaluation pipeline, including the main HumanEval benchmark, cross-model ablation (Qwen), model scaling ablation, and the controlled Oracle mechanistic experiment.

## Overview
- **Dataset**: [HumanEval](https://huggingface.co/datasets/openai/openai_humaneval) — 164 Python programming tasks
- **Pipelines**: Single-Agent, 2-Agent (Architect + Implementer), 3-Agent (+ Validator)
- **Models**: DeepSeek-Coder-1.3B, DeepSeek-Coder-6.7B, Qwen2.5-Coder-3B, CodeLlama-7B
- **Metrics**: Pass@1, Coordination Collapse Rate, Spec Defect Analysis
- **Output**: Per-task results, summary CSVs, figures, and mechanistic experiment results

## Step 1: Clone the Repository

In [ ]:
import os
import subprocess
from pathlib import Path

repo_url = "https://github.com/pris25123/Coordination-Collapse.git"
target_dir = Path.home() / "Coordination-Collapse"

if not target_dir.exists():
    print(f"Cloning repository to {target_dir}...")
    result = subprocess.run(["git", "clone", repo_url, str(target_dir)], capture_output=True, text=True)
    if result.returncode == 0:
        print("Repository cloned successfully!")
    else:
        print("Error cloning repository:")
        print(result.stderr)
else:
    print(f"Repository already exists at {target_dir}")

print(f"\nRepository location: {target_dir}")

## Step 2: Verify Repository Structure

In [ ]:
print("Repository structure:")
print("=" * 50)

for root, dirs, files in os.walk(target_dir):
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    level = root.replace(str(target_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")
    if level > 2:
        break

## Step 3: Install Dependencies

In [ ]:
print("Installing dependencies...")
print("=" * 50)

result = subprocess.run(
    ["pip", "install", "transformers", "bitsandbytes", "torch", "scipy", "numpy",
     "datasets", "pandas", "matplotlib", "seaborn", "tqdm"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("Dependencies installed successfully!")
else:
    print("Error installing dependencies:")
    print(result.stderr)

## Step 4: Run Main Experiment (Single-Agent, 2-Agent, 3-Agent on HumanEval)

Runs the full HumanEval benchmark across all three pipeline configurations using DeepSeek-Coder-1.3B.
- Single-Agent: code directly from problem
- 2-Agent: Architect generates spec, Implementer writes code from spec only
- 3-Agent: adds a Validator + repair loop

In [ ]:
exp1_dir = target_dir / "Experiment1"
exp1_notebook = exp1_dir / "experiment1.ipynb"

print("Running Experiment 1: Main HumanEval Benchmark")
print("=" * 60)
print("This will:")
print("  1. Load HumanEval (164 tasks)")
print("  2. Run Single-Agent baseline (DeepSeek-Coder-1.3B)")
print("  3. Run 2-Agent pipeline (Architect + Implementer)")
print("  4. Run 3-Agent pipeline (+ Validator)")
print("  5. Compute Pass@1, collapse rate, 95% CIs, McNemar's test")
print("  6. Save per-task results and summary CSVs")
print("=" * 60)

result = subprocess.run(
    ["jupyter", "nbconvert", "--to", "notebook", "--execute",
     "--inplace", str(exp1_notebook)],
    capture_output=True, text=True, timeout=3600
)

print(result.stdout)
if result.stderr:
    print("Warnings/Errors:")
    print(result.stderr)

if result.returncode == 0:
    print("\nExperiment 1 completed successfully!")
else:
    print(f"\nExperiment 1 ended with return code: {result.returncode}")

## Step 5: Run Ablation 1 — Model Scaling (DeepSeek 1.3B vs 6.7B)

Tests whether scaling all agent roles to a larger model reduces coordination collapse.

In [ ]:
ablation1_notebook = exp1_dir / "ablation1" / "ablation1.ipynb"

print("Running Ablation 1: Model Scaling (1.3B vs 6.7B)")
print("=" * 60)

result = subprocess.run(
    ["jupyter", "nbconvert", "--to", "notebook", "--execute",
     "--inplace", str(ablation1_notebook)],
    capture_output=True, text=True, timeout=3600
)

print(result.stdout)
if result.stderr:
    print("Warnings/Errors:")
    print(result.stderr)

if result.returncode == 0:
    print("\nAblation 1 completed successfully!")
else:
    print(f"\nAblation 1 ended with return code: {result.returncode}")

## Step 6: Run Ablation 2 — Cross-Model (Qwen2.5-Coder-3B)

Confirms that coordination collapse is not an artifact of a specific model family.

In [ ]:
ablation2_notebook = exp1_dir / "ablation2" / "ablation2_qwen_agents.ipynb"

print("Running Ablation 2: Cross-Model (Qwen2.5-Coder-3B)")
print("=" * 60)

result = subprocess.run(
    ["jupyter", "nbconvert", "--to", "notebook", "--execute",
     "--inplace", str(ablation2_notebook)],
    capture_output=True, text=True, timeout=3600
)

print(result.stdout)
if result.stderr:
    print("Warnings/Errors:")
    print(result.stderr)

if result.returncode == 0:
    print("\nAblation 2 completed successfully!")
else:
    print(f"\nAblation 2 ended with return code: {result.returncode}")

## Step 7: Run Experiment 2 — Oracle Mechanistic Experiment

Isolates the root cause of collapse using three conditions:
- **Direct**: model solves from problem statement
- **Hierarchical**: model solves from spec only (no original problem)
- **Oracle**: model solves from both spec and original problem

Oracle recovery = information loss, not bad planning.

In [ ]:
exp2_dir = target_dir / "Experiment2"
exp2_notebook = exp2_dir / "Experiment2.ipynb"

print("Running Experiment 2: Oracle Mechanistic Experiment")
print("=" * 60)
print("Conditions: Direct | Hierarchical | Oracle")
print("Models: DeepSeek-Coder-1.3B, CodeLlama-7B")
print("=" * 60)

result = subprocess.run(
    ["jupyter", "nbconvert", "--to", "notebook", "--execute",
     "--inplace", str(exp2_notebook)],
    capture_output=True, text=True, timeout=3600
)

print(result.stdout)
if result.stderr:
    print("Warnings/Errors:")
    print(result.stderr)

if result.returncode == 0:
    print("\nExperiment 2 completed successfully!")
else:
    print(f"\nExperiment 2 ended with return code: {result.returncode}")

## Step 8: Load and Display Results

In [ ]:
import pandas as pd
import json

results_dir = exp1_dir / "experiment1_results"
summary_file = results_dir / "results_summary.csv"

if summary_file.exists():
    df = pd.read_csv(summary_file)
    print("EXPERIMENT 1 — MAIN RESULTS SUMMARY")
    print("=" * 60)
    print(df.to_string(index=False))
else:
    print(f"Summary file not found at {summary_file}")

# Ablation 2 summary
abl2_summary = exp1_dir / "ablation2" / "ablation2_results" / "results_summary.csv"
if abl2_summary.exists():
    df2 = pd.read_csv(abl2_summary)
    print("\nABLATION 2 — QWEN CROSS-MODEL SUMMARY")
    print("=" * 60)
    print(df2.to_string(index=False))

# Experiment 2 summary
exp2_summary = exp2_dir / "results" / "results_unified_summary.json"
if exp2_summary.exists():
    with open(exp2_summary) as f:
        exp2_data = json.load(f)
    print("\nEXPERIMENT 2 — ORACLE MECHANISTIC RESULTS")
    print("=" * 60)
    print(json.dumps(exp2_data, indent=2))

## Step 9: Display Figures

In [ ]:
from IPython.display import Image, display

figures_dir = exp1_dir / "experiment1_figures"
figure_labels = {
    "figure1_deepseek_pass1.png": "Fig 1: Pass@1 — Single vs 2-Agent vs 3-Agent (DeepSeek-1.3B)",
    "figure2_qwen_ablation.png": "Fig 2: Cross-model comparison — DeepSeek vs Qwen",
    "figure3_oracle_experiment.png": "Fig 3: Oracle experiment — Direct vs Hierarchical vs Oracle",
    "figure4_difficulty_bands.png": "Fig 4: Pass@1 by difficulty band",
    "figure5_validator_analysis.png": "Fig 5: Validator verdict distribution and revision outcomes",
    "figure6_spec_defects.png": "Fig 6: Specification defects in collapse cases",
}

for filename, label in figure_labels.items():
    fig_path = figures_dir / filename
    if fig_path.exists():
        print(f"\n{label}")
        display(Image(filename=str(fig_path), width=600))
    else:
        print(f"Figure not found: {filename}")

# Experiment 2 figures
exp2_figs_dir = exp2_dir / "experiment2_figures"
for fig_path in sorted(exp2_figs_dir.glob("*.png")):
    print(f"\n{fig_path.stem}")
    display(Image(filename=str(fig_path), width=600))

## Step 10: Verify Output Files

In [ ]:
print("Generated Output Files")
print("=" * 60)

output_dirs = [
    ("Experiment 1 Results", exp1_dir / "experiment1_results"),
    ("Ablation 2 Results", exp1_dir / "ablation2" / "ablation2_results"),
    ("Experiment 2 Results", exp2_dir / "results"),
    ("Experiment 1 Figures", exp1_dir / "experiment1_figures"),
    ("Experiment 2 Figures", exp2_dir / "experiment2_figures"),
]

for label, d in output_dirs:
    if d.exists():
        files = list(d.iterdir())
        print(f"\n{label} ({len(files)} files):")
        for f in sorted(files):
            size = f.stat().st_size / 1024
            print(f"  {f.name} ({size:.1f} KB)")
    else:
        print(f"\n{label}: directory not found")

## Summary

This notebook ran the complete evaluation for **The Coordination Tax** paper:

1. **Cloned** the Coordination-Collapse repository
2. **Verified** repository structure
3. **Installed** all dependencies
4. **Experiment 1**: Single, 2-Agent, 3-Agent pipelines on HumanEval (DeepSeek-1.3B)
5. **Ablation 1**: Model scaling (1.3B vs 6.7B) — collapse unchanged
6. **Ablation 2**: Cross-model with Qwen-3B — collapse persists
7. **Experiment 2**: Oracle mechanistic experiment — collapse traced to information loss
8. **Displayed** results and figures
9. **Verified** all output files

### Key Finding
Introducing an intermediate Architect agent causes a **47.6% drop in Pass@1** for DeepSeek and **43.3%** for Qwen. Adding a Validator or scaling model size does not fix this. The Oracle experiment confirms the root cause is **information loss at the agent boundary**, not bad planning or insufficient model capacity.